# Build a RAG App Over Your Own Notes — Colab / Kaggle / Binder companion

This notebook mirrors the local `uv` project from the course's
[Build a RAG App Over Your Own Notes](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/rag-notes)
lesson, adapted to run in a hosted notebook with no local files: chunking, local
embeddings with `sentence-transformers`, NumPy cosine-similarity retrieval, and a final
generation step against a free-tier LLM.

See the [lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/rag-notes) for the full walkthrough and the
[local example project](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/rag-notes) for the real, file-based version of this same code.

## Step 0: Install dependencies

In [ ]:
!pip install sentence-transformers numpy openai

## Step 1: Sample notes

A hosted notebook has no local `notes/` folder to read from, so the same three sample
notes shipped with the local example are embedded here directly as strings.

In [ ]:
NOTES = {
    "course-structure.md": """## Course structure

The course has two required sections, taught entirely in the browser via
JupyterLite: Python 101, covering variables, loops, functions, and basic
data structures, and Data Analysis, covering pandas, DataFrames, and
groupby-style aggregation.

Python 101 has two tracks: a Normal track for beginners and a Hard track
that goes deeper, including building a tiny language model from scratch out
of word counts and bigram probabilities.

After both sections, there is an optional, ungraded "Real-World Projects"
section. Unlike the rest of the course, these projects install Python for
real on your own machine using `uv`, instead of running in a browser
sandbox.""",
    "pandas-groupby.md": """## pandas groupby, briefly

`groupby` splits a DataFrame into groups based on the values in one or more
columns, then lets you compute a summary statistic separately for each
group, then (usually) combines the results back into a single DataFrame or
Series. This three-step shape is often called split-apply-combine.

```python
import pandas as pd

df = pd.DataFrame({
    "city": ["Rabat", "Rabat", "Fes", "Fes"],
    "sales": [100, 150, 80, 90],
})

df.groupby("city")["sales"].mean()
```

This returns the average sales per city, one row per unique value in the
`city` column. The same pattern works with `.sum()`, `.count()`, `.max()`,
or a custom function passed to `.agg()`.""",
    "python-tip-mutability.md": """## Python tip: mutable default arguments

A classic Python pitfall: never use a mutable object, like a list or dict,
as a default argument value.

```python
def add_item(item, items=[]):  # BUG: the same list is reused across calls
    items.append(item)
    return items
```

Default argument values are evaluated once, when the function is defined,
not once per call. Every call that doesn't pass `items` explicitly shares
the exact same list object, so items silently pile up across unrelated
calls.

The fix is to default to `None` and create a fresh list inside the
function body:

```python
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items
```""",
}

print(f"Loaded {len(NOTES)} sample notes: {list(NOTES.keys())}")

## Step 2: Split notes into chunks

Same chunking logic as [`prepare_notes.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/rag-notes/prepare_notes.py) in the local example: split on blank
lines into paragraphs, then greedily re-merge short paragraphs up to a target size, so
retrieval later gets focused-but-not-tiny chunks instead of a whole file at once.

In [ ]:
TARGET_CHUNK_SIZE = 500  # characters


def split_into_paragraphs(text: str) -> list[str]:
    """Splits on blank lines, dropping empty paragraphs."""
    paragraphs = [p.strip() for p in text.split("\n\n")]
    return [p for p in paragraphs if p]


def merge_short_paragraphs(paragraphs: list[str], target_size: int) -> list[str]:
    """Greedily merges consecutive short paragraphs up to target_size characters."""
    chunks = []
    current = ""
    for paragraph in paragraphs:
        if current and len(current) + len(paragraph) > target_size:
            chunks.append(current)
            current = paragraph
        else:
            current = f"{current}\n\n{paragraph}" if current else paragraph
    if current:
        chunks.append(current)
    return chunks


def load_chunks(notes: dict[str, str]) -> list[dict]:
    """Returns a list of {"text": ..., "source": ...} dicts, one per chunk,
    across every note in `notes` (name -> full text)."""
    chunks = []
    for source, text in notes.items():
        paragraphs = split_into_paragraphs(text)
        for chunk_text in merge_short_paragraphs(paragraphs, TARGET_CHUNK_SIZE):
            chunks.append({"text": chunk_text, "source": source})
    return chunks


chunks = load_chunks(NOTES)
print(f"Loaded {len(chunks)} chunks from {len(NOTES)} notes")
for chunk in chunks[:3]:
    preview = chunk["text"][:80].replace("\n", " ")
    print(f"  [{chunk['source']}] {preview}...")

## Step 3: Embed the chunks locally

Same model as [`build_index.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/rag-notes/build_index.py) in the local example: `all-MiniLM-L6-v2`, run entirely
on CPU, no API key needed. Since this notebook has no `index.npy`/`chunks.json` files to
persist to, the embeddings are just kept in memory for the rest of the notebook.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Embedding {len(chunks)} chunks with {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts, normalize_embeddings=True)

print(f"Embedded {embeddings.shape[0]} chunks ({embeddings.shape[1]}-dim)")

## Step 4: Retrieve relevant chunks

Same cosine-similarity retrieval as [`retrieve.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/rag-notes/retrieve.py) — since every embedding is already
normalized to length 1, cosine similarity collapses to a plain dot product.

In [ ]:
def retrieve(question: str, top_k: int = 3) -> list[dict]:
    """Returns the top_k chunks most similar to `question`, each with its
    similarity score, ranked highest first."""
    question_vector = model.encode([question], normalize_embeddings=True)[0]
    similarities = embeddings @ question_vector
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [
        {**chunks[i], "score": float(similarities[i])}
        for i in top_indices
    ]


results = retrieve("What is this course about?")
for r in results:
    print(f"{r['score']:.3f}  [{r['source']}]  {r['text'][:80]}...")

## Step 5: Get a free-tier LLM API key

Pick any provider from the table in the [lesson's Step 5](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/rag-notes#step-5-generate-an-answer-with-a-free-llm) —
GitHub Models is the suggested default since it needs no separate signup. Paste the key
below; `getpass` keeps it out of the notebook's saved output and cell history.

In [ ]:
import getpass
import os

os.environ["GITHUB_TOKEN"] = getpass.getpass("Paste your GitHub Models API key: ")

## Step 6: Generate an answer with a free LLM

Same prompt and client setup as [`ask.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/rag-notes/ask.py) — GitHub Models exposes an
OpenAI-compatible API, so the plain `openai` client library works without any extra package.

In [ ]:
from openai import OpenAI

PROMPT_TEMPLATE = """Answer the question using ONLY the context below. If the
context doesn't contain the answer, say so -- do not make something up.

Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question: str, chunks: list[dict]) -> str:
    context = "\n\n".join(f"[{c['source']}] {c['text']}" for c in chunks)
    return PROMPT_TEMPLATE.format(context=context, question=question)


def ask(question: str, top_k: int = 3) -> str:
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)

    client = OpenAI(
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference",
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # confirm this still has a free tier before running
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

## Try it: ask a real question

In [ ]:
question = "What are the two required sections of the course, and what does each cover?"
answer = ask(question)
print(f"Q: {question}\n")
print(f"A: {answer}")